<a href="https://colab.research.google.com/github/Saliyah-53/saliyah-stanceeval2026/blob/main/SaliAI_Hashtag_Stance_Track1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SaliAI — Hashtag Idea: Extracting Stance from Explicit Hashtags (Track 1)

Some hashtags carry the stance literally (e.g., #لن_تقودي = Against,
#سنسوق_فوق_خشومكم = Favor). We build a small stance-hashtag lexicon and use it as a
priority over Fanar's prediction when an explicit hashtag is present.

We try two variants:
- **hashtag_override:** if a hashtag has an explicit stance -> use it (overrides Fanar)
- **hashtag_rescue_none:** use the hashtag only when Fanar predicts None (more conservative)

Requires: `sub_fanar_only.txt` (Fanar predictions from an earlier notebook) and
`test_seen.csv`. No GPU needed -- text logic only.

In [ ]:
import pandas as pd, re, os, zipfile
DATA_DIR="./data"; OUT="./hashtag_outputs"; os.makedirs(OUT, exist_ok=True)
TEXT_COL="text"; TARGET_COL="target"

# قاموس الهاشتاقات الموقفية (مبني من تحليل بيانات الاختبار الفعلية)
# نطبّع الهاشتاق: نحذف # و التشكيل و نوحّد الألف/الهمزات لالتقاط التهجئات المختلفة
def norm_tag(h):
    h=h.replace("#","").replace("_","")
    h=re.sub(r"[أإآ]","ا",h); h=re.sub(r"ة","ه",h); h=re.sub(r"ى","ي",h)
    return h

HASHTAG_STANCE = {

    "لن تقودي": "Against",
    "سنسوق فوق خشومكم": "Favor",
    "ستقودي والشعب معك": "Favor",
    "الملك ينتصرلقياده": "Favor",
    "الملك ينتصرلقياده المراه": "Favor",
    "المجتمع مع قياده": "Favor",
    "السماح بقياده": "Favor",
    "قياده المراه ضروره": "Favor",
    "قياده المراه للسياره ضروره": "Favor",
}
def norm_key(k):
    k=re.sub(r"[أإآ]","ا",k); k=re.sub(r"ة","ه",k); k=re.sub(r"ى","ي",k); return k
HASHTAG_STANCE={norm_key(k):v for k,v in HASHTAG_STANCE.items()}

def hashtag_stance(text):
    """يرجع Favor/Against إذا وُجد هاشتاق موقفه صريح، وإلا None-signal."""
    for h in re.findall(r"#[\w_]+", str(text)):
        n=norm_tag(h)
        for key,st in HASHTAG_STANCE.items():
            if key in n:
                return st
    return None


df=pd.read_csv(f"{DATA_DIR}/test_seen.csv", keep_default_na=False)
if "tweet_text" in df.columns and TEXT_COL not in df.columns: df=df.rename(columns={"tweet_text":TEXT_COL})
hstance=[hashtag_stance(t) for t in df[TEXT_COL]]
from collections import Counter
print("توزيع إشارة الهاشتاق:", Counter([h if h else "لا-إشارة" for h in hstance]))
print("عدد التغريدات ذات الهاشتاق الصريح:", sum(1 for h in hstance if h))

توزيع إشارة الهاشتاق: Counter({'Against': 167, 'لا-إشارة': 159, 'Favor': 26})
عدد التغريدات ذات الهاشتاق الصريح: 193


In [ ]:

FANAR_TXT=None
for p in ["./sub_fanar_only.txt", f"{OUT}/sub_fanar_only.txt", "/content/sub_fanar_only.txt"]:
    if os.path.exists(p): FANAR_TXT=p; break
assert FANAR_TXT, "ارفع sub_fanar_only.txt أولاً (تنبؤات Fanar على Track 1)"
fanar=[x for x in open(FANAR_TXT, encoding="utf-8").read().split("\n") if x]
assert len(fanar)==len(df), f"عدم تطابق: fanar={len(fanar)} vs test={len(df)}"
print("Fanar preds:", Counter(fanar))

Fanar preds: Counter({'Against': 214, 'Favor': 120, 'None': 18})


In [ ]:

def save_sub(preds, name):
    txt=f"{OUT}/{name}.txt"; open(txt,"w",encoding="utf-8").write("\n".join(preds)+"\n")
    zp=f"{OUT}/{name}.zip"
    with zipfile.ZipFile(zp,"w",zipfile.ZIP_DEFLATED) as z: z.write(txt, arcname="submission_seen.txt")
    print(f"{name}: {Counter(preds)}  -> {zp}")
    return zp


override=[hstance[i] if hstance[i] else fanar[i] for i in range(len(df))]
save_sub(override, "sub_hashtag_override")


rescue=[hstance[i] if (fanar[i]=="None" and hstance[i]) else fanar[i] for i in range(len(df))]
save_sub(rescue, "sub_hashtag_rescue_none")


d1=sum(1 for i in range(len(df)) if override[i]!=fanar[i])
d2=sum(1 for i in range(len(df)) if rescue[i]!=fanar[i])
print(f"\nتغييرات override: {d1} | rescue_none: {d2}")
print(">>> ارفع النسختين على Track 1 وقارن مع Fanar (0.7152) <<<")

sub_hashtag_override: Counter({'Against': 215, 'Favor': 119, 'None': 18})  -> ./hashtag_outputs/sub_hashtag_override.zip
sub_hashtag_rescue_none: Counter({'Against': 214, 'Favor': 120, 'None': 18})  -> ./hashtag_outputs/sub_hashtag_rescue_none.zip

تغييرات override: 23 | rescue_none: 0
>>> ارفع النسختين على Track 1 وقارن مع Fanar (0.7152) <<<
